# Preprocessing and ball repair

More than a Hack 2027, Phase 1, ball possession. The input is the automatic tracking (ATD) of the tactical video: one
row per frame, 199,319 rows at 25 fps. The output is a cleaned tracking object, cached in
`artifacts/tracking_clean.npz`, with:

* smoothed player tracks (a Gaussian filter that skips missing values) and velocities;
* a repaired ball track. We remove spikes. Gaps up to 50 frames get a cubic spline, or a straight line when the
  spline would go faster than 35 m/s, and longer gaps take the centre of the densest group of moving players. A flag on
  each frame records where its ball position came from.

We process each half on its own (frames 11,185-90,651 and 110,808-190,197), so no interpolation crosses half time.

This notebook only holds definitions. The run notebooks load its cells tagged `defs`.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent   # the repository root: notebooks run from possession/
PIPELINE_DIR = REPO_ROOT / "possession"
ARTIFACT_DIR = PIPELINE_DIR / "artifacts"
OUTPUT_DIR = PIPELINE_DIR / "outputs"     

def load_notebook_defs(filename: str, namespace: dict) -> None:
    loaded = namespace.setdefault("_LOADED_NOTEBOOKS", set())
    if filename in loaded:
        return
    loaded.add(filename)
    nb = json.loads((PIPELINE_DIR / filename).read_text(encoding="utf-8"))
    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] == "code" and "defs" in cell.get("metadata", {}).get("tags", []):
            exec(compile("".join(cell["source"]), f"{filename}::cell{i}", "exec"), namespace)

In [ ]:
import hashlib
import os
import re
import time
from dataclasses import asdict, dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy import ndimage
from scipy.interpolate import CubicSpline

GAME = "BAR-ATH"
LABEL_TO_ID = {"H": 0, "A": 1, "D": 2}

FALLBACK_N_FRAMES = 199_319
FALLBACK_HALF_BLOCKS = [(1, 11_185, 90_651), (2, 110_808, 190_197)]  # (half, first, last) video frames, inclusive
FALLBACK_PITCH_M = (105.3, 68.0)
# FAQ.md: the 2nd half is graded from frame 110,808 but only kicks off at 113,742 -> those 2,934 graded frames are all 'D'.
REAL_KNOWN_DEAD_RANGES = [(110_808, 113_741)]


def _import_games():
    """Import the repo's games.py (asset paths, half boundaries) if it is next to possession/."""
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    try:
        import games  # type: ignore
        return games
    except Exception:
        return None


def get_half_blocks() -> List[Tuple[int, int, int]]:
    games = _import_games()
    if games is not None:
        try:
            return [tuple(int(v) for v in blk) for blk in games.half_video_blocks(GAME)]
        except Exception:
            pass
    return list(FALLBACK_HALF_BLOCKS)


def get_paths() -> Dict[str, Path]:
    """Every provided file (tactical source) + the public ground truth. Existence is NOT checked here."""
    games = _import_games()
    if games is not None:
        paths = {k: Path(v) for k, v in games.asset_paths().items()}
    else:
        data_dir = Path(os.environ.get("MTAH_DATA_DIR") or (REPO_ROOT / "data"))
        stem = data_dir / "tactical" / "BAR-ATH_tactical"
        paths = {
            "atd_meta": Path(str(stem) + "_atd.xml"),
            "atd_raw": Path(str(stem) + "_atd.txt"),
        }
    # The public 5-minute ground truth ships INSIDE the repo (evaluation/gt/), not in data/.
    paths["gt_public"] = REPO_ROOT / "evaluation" / "gt" / "BAR-ATH_possession_gt_public.csv"
    return paths


def require_files(paths: Dict[str, Path], keys: Sequence[str]) -> None:
    """Raise FileNotFoundError listing every missing file (the pipeline only runs on the real data)."""
    missing = [str(paths[k]) for k in keys if not paths[k].is_file()]
    if missing:
        raise FileNotFoundError(
            "Required data files not found:\n  " + "\n  ".join(missing) +
            "\nUnpack the downloaded bundle into data/ (see DATA.md) or set MTAH_DATA_DIR. "
            "The public ground truth ships with the repo in evaluation/gt/.")


def file_signature(*paths: Path) -> str:
    """Cheap cache key: size + mtime of the given files."""
    parts = []
    for p in paths:
        st = Path(p).stat()
        parts.append(f"{st.st_size}-{st.st_mtime_ns}")
    return "|".join(parts)


@dataclass
class PreprocessConfig:
    fps: int = 25
    pitch_m: Tuple[float, float] = FALLBACK_PITCH_M
    # --- ball infilling (spec: spline <= 50 frames, <= 35 m/s, cluster fallback beyond) ---
    max_spline_gap: int = 50
    max_ball_speed_mps: float = 35.0
    spline_context_points: int = 4          # observed samples used on each side of a gap
    reject_observed_spikes: bool = True     # drop observed ball jumps faster than max_ball_speed_mps
    ball_margin_m: float = 5.0              # infilled positions are clipped to the pitch +/- this margin
    cluster_radius_m: float = 10.0
    cluster_min_speed_mps: float = 1.0      # "moving" player threshold for the cluster fallback
    cluster_smooth_sigma: float = 8.0       # frames
    cluster_blend_frames: int = 12          # ramp from the last/next observed ball into the cluster estimate
    # --- player tracks ---
    player_sigma_frames: float = 2.0
    max_player_speed_mps: float = 12.0      # faster than this = identity switch / glitch -> velocity set to NaN


def nan_gaussian(arr: np.ndarray, sigma: float, axis: int = 0) -> np.ndarray:
    """Gaussian smoothing that ignores NaNs (normalised convolution). Missing samples stay NaN."""
    a = np.asarray(arr, dtype=np.float32)
    if sigma <= 0:
        return a.copy()
    m = np.isfinite(a)
    num = ndimage.gaussian_filter1d(np.where(m, a, 0.0).astype(np.float32), sigma, axis=axis, mode="nearest", truncate=3.0)
    den = ndimage.gaussian_filter1d(m.astype(np.float32), sigma, axis=axis, mode="nearest", truncate=3.0)
    out = np.where(m & (den > 1e-3), num / np.maximum(den, 1e-3), np.nan)
    return out.astype(np.float32)

In [ ]:
@dataclass
class TrackingData:
    """Raw tracking, indexed by VIDEO frame (row i of a submission == row i here)."""
    px: np.ndarray                 # (T, P) float32 metres, NaN = missing
    py: np.ndarray                 # (T, P)
    team: np.ndarray               # (P,) int8: 0 = home, 1 = away
    ball_xy: np.ndarray            # (T, 2) float32 metres, NaN = missing
    half_blocks: List[Tuple[int, int, int]]
    fps: int = 25
    pitch_m: Tuple[float, float] = FALLBACK_PITCH_M
    known_dead_ranges: List[Tuple[int, int]] = field(default_factory=list)
    signature: str = ""

    @property
    def n_frames(self) -> int:
        return int(self.px.shape[0])

    @property
    def n_slots(self) -> int:
        return int(self.px.shape[1])

    def half_id(self) -> np.ndarray:
        """(T,) int8: 1 / 2 inside the graded halves, 0 elsewhere (before kick-off, half time, after the whistle)."""
        h = np.zeros(self.n_frames, np.int8)
        for half, a, b in self.half_blocks:
            h[a:b + 1] = half
        return h

    def save(self, path: Path) -> None:
        np.savez_compressed(
            path, px=self.px, py=self.py, team=self.team, ball_xy=self.ball_xy,
            half_blocks=np.asarray(self.half_blocks, np.int64),
            known_dead=np.asarray(self.known_dead_ranges, np.int64).reshape(-1, 2),
            fps=np.asarray(self.fps), pitch_m=np.asarray(self.pitch_m, np.float64),
            signature=np.asarray(self.signature))

    @classmethod
    def load(cls, path: Path) -> "TrackingData":
        z = np.load(path, allow_pickle=False)
        return cls(
            px=z["px"], py=z["py"], team=z["team"], ball_xy=z["ball_xy"],
            half_blocks=[tuple(int(v) for v in r) for r in z["half_blocks"]],
            fps=int(z["fps"]), pitch_m=tuple(float(v) for v in z["pitch_m"]),
            known_dead_ranges=[tuple(int(v) for v in r) for r in z["known_dead"]],
            signature=str(z["signature"]))


_SLOT_COL = re.compile(r"^(?P<pid>.+)_(?P<kind>x|y|s|d|z)$")


def tracking_from_kloppy(ds, n_frames: int, half_blocks, pitch_m=FALLBACK_PITCH_M, fps: int = 25) -> TrackingData:
    """Turn a kloppy TrackingDataset (Metrica EPTS) into a TrackingData aligned to VIDEO frames.

    DATA.md: ATD frame_id runs 1..199319 and ``video_frame = frame_id - 1``; coordinates are normalised 0-1;
    columns are ``<slot>_x`` / ``<slot>_y`` / ``<slot>_s`` plus ``ball_x`` / ``ball_y``; missing values are NaN.
    """
    df = ds.to_df()
    if not isinstance(df, pd.DataFrame):          # newer kloppy versions can return polars
        df = df.to_pandas()

    team_of: Dict[str, int] = {}
    for team in ds.metadata.teams:
        ground = str(getattr(team.ground, "value", team.ground)).lower()
        for pl in team.players:
            team_of[str(pl.player_id)] = 0 if "home" in ground else 1

    slots: List[str] = []
    for c in df.columns:
        m = _SLOT_COL.match(str(c))
        if m and m.group("kind") == "x" and m.group("pid") != "ball":
            slots.append(m.group("pid"))
    unmapped = [s for s in slots if s not in team_of]
    if not slots or unmapped:
        raise RuntimeError(
            "Could not map tracking columns to teams.\n"
            f"  slots found in columns : {slots[:6]} ...\n"
            f"  player ids in metadata : {list(team_of)[:6]} ...\n"
            "Inspect `ds.metadata.teams` and `ds.to_df().columns` and adapt `tracking_from_kloppy`.")

    video_frame = pd.to_numeric(df["frame_id"], errors="coerce").to_numpy(np.float64) - 1
    keep = np.isfinite(video_frame) & (video_frame >= 0) & (video_frame < n_frames)
    vf = video_frame[keep].astype(np.int64)

    def col(name: str) -> np.ndarray:
        return pd.to_numeric(df[name], errors="coerce").to_numpy(np.float32)[keep]

    P = len(slots)
    px = np.full((n_frames, P), np.nan, np.float32)
    py = np.full((n_frames, P), np.nan, np.float32)
    for j, s in enumerate(slots):
        px[vf, j] = col(f"{s}_x")
        py[vf, j] = col(f"{s}_y")
    ball = np.full((n_frames, 2), np.nan, np.float32)
    ball[vf, 0] = col("ball_x")
    ball[vf, 1] = col("ball_y")

    # normalised 0-1 -> metres (if the file is already in metres the 99th percentile is far above 1.5)
    if np.nanpercentile(px[::50], 99) <= 1.5:
        L, W = pitch_m
        px *= L
        py *= W
        ball[:, 0] *= L
        ball[:, 1] *= W
    team = np.array([team_of[s] for s in slots], dtype=np.int8)
    return TrackingData(px=px, py=py, team=team, ball_xy=ball, half_blocks=list(half_blocks), fps=fps,
                        pitch_m=tuple(pitch_m), known_dead_ranges=list(REAL_KNOWN_DEAD_RANGES))


def load_atd_tracking(paths: Optional[Dict[str, Path]] = None, use_cache: bool = True, verbose: bool = True) -> TrackingData:
    """Load BAR-ATH_tactical_atd.{xml,txt} with kloppy (slow the first time -> cached in artifacts/)."""
    paths = paths or get_paths()
    require_files(paths, ("atd_meta", "atd_raw"))
    ARTIFACT_DIR.mkdir(exist_ok=True)
    cache = ARTIFACT_DIR / "raw_tracking_real.npz"
    sig = file_signature(paths["atd_meta"], paths["atd_raw"])
    if use_cache and cache.is_file():
        cached = TrackingData.load(cache)
        if cached.signature == sig:
            if verbose:
                print(f"[raw] loaded cached tracking {cache.name}")
            return cached
    from kloppy import metrica  # imported lazily: only needed for real data
    t0 = time.time()
    if verbose:
        print("[raw] parsing ATD with kloppy (several minutes and a few GB of RAM the first time) ...")
    ds = metrica.load_tracking_epts(meta_data=str(paths["atd_meta"]), raw_data=str(paths["atd_raw"]))
    td = tracking_from_kloppy(ds, FALLBACK_N_FRAMES, get_half_blocks())
    td.signature = sig
    td.save(cache)
    if verbose:
        print(f"[raw] done in {time.time() - t0:.0f}s -> {td.n_frames} frames, {td.n_slots} slots, cached.")
    return td

In [ ]:
class BallInfiller:
    """Repairs the ball track of ONE contiguous block (one half).

    1. observed jumps faster than ``max_ball_speed_mps`` are removed (tracker glitches);
    2. gaps <= ``max_spline_gap`` frames with an observed ball on both sides -> cubic spline through nearby observed samples;
       if the spline would make the ball faster than the physical limit the gap is filled linearly instead (= truncating
       the overshoot) and flagged as such;
    3. longer gaps (and gaps touching the block edge) -> centroid of the densest cluster of *moving* players, smoothed and
       blended into the last / next observed ball position so the track has no jump.
    """
    SRC_OBSERVED, SRC_SPLINE, SRC_LINEAR, SRC_CLUSTER, SRC_NONE = 0, 1, 2, 3, 4

    def __init__(self, cfg: PreprocessConfig):
        self.cfg = cfg

    def _reject_spikes(self, xy: np.ndarray) -> int:
        cfg = self.cfg
        max_step = cfg.max_ball_speed_mps / cfg.fps                      # metres per frame
        obs = np.flatnonzero(np.isfinite(xy).all(axis=1))
        if len(obs) < 2:
            return 0
        rejected, streak, last = 0, 0, obs[0]
        for i in obs[1:]:
            allowed = max_step * (i - last) + 1.0                         # +1 m tolerance for tracking noise
            if np.hypot(xy[i, 0] - xy[last, 0], xy[i, 1] - xy[last, 1]) > allowed and streak < 10:
                xy[i] = np.nan
                rejected += 1
                streak += 1
                continue
            streak, last = 0, i
        return rejected

    @staticmethod
    def _gaps(missing: np.ndarray) -> List[Tuple[int, int]]:
        if not missing.any():
            return []
        d = np.diff(np.concatenate([[0], missing.astype(np.int8), [0]]))
        return list(zip(np.flatnonzero(d == 1).tolist(), np.flatnonzero(d == -1).tolist()))

    def _cluster_centroids(self, px, py, speed, idx: np.ndarray) -> np.ndarray:
        """Per frame: centroid of the densest group of moving players (all valid players if fewer than 3 are moving)."""
        cfg = self.cfg
        R2 = cfg.cluster_radius_m ** 2
        out = np.full((len(idx), 2), np.nan)
        for s in range(0, len(idx), 1024):
            sl = idx[s:s + 1024]
            x, y, sp = px[sl], py[sl], speed[sl]
            valid = np.isfinite(x) & np.isfinite(y)
            moving = valid & (np.nan_to_num(sp, nan=0.0) >= cfg.cluster_min_speed_mps)
            use = np.where(moving.sum(1, keepdims=True) >= 3, moving, valid)
            dx = x[:, :, None] - x[:, None, :]
            dy = y[:, :, None] - y[:, None, :]
            near = ((dx * dx + dy * dy) < R2) & use[:, None, :] & use[:, :, None]
            dens = np.where(use, near.sum(2), -1)
            best = dens.argmax(1)
            member = near[np.arange(len(sl)), best]
            cnt = member.sum(1)
            ok = cnt > 0
            cx = np.where(member, np.nan_to_num(x), 0.0).sum(1) / np.maximum(cnt, 1)
            cy = np.where(member, np.nan_to_num(y), 0.0).sum(1) / np.maximum(cnt, 1)
            blk = out[s:s + len(sl)]
            blk[ok, 0], blk[ok, 1] = cx[ok], cy[ok]
        return out

    def _finish_cluster_segment(self, seg: np.ndarray, left, right, lo, hi) -> np.ndarray:
        cfg = self.cfg
        m = len(seg)
        good = np.isfinite(seg).all(axis=1)
        if not good.any():
            return seg
        t = np.arange(m)
        seg = np.column_stack([np.interp(t, t[good], seg[good, j]) for j in (0, 1)])
        seg = ndimage.gaussian_filter1d(seg, cfg.cluster_smooth_sigma, axis=0, mode="nearest")
        bl = max(cfg.cluster_blend_frames, 1)
        if left is not None:
            wl = np.clip((t + 1) / bl, 0, 1)[:, None]
            seg = wl * seg + (1 - wl) * left
        if right is not None:
            wr = np.clip((m - t) / bl, 0, 1)[:, None]
            seg = wr * seg + (1 - wr) * right
        return np.clip(seg, lo, hi)

    def infill_block(self, ball_xy: np.ndarray, px: np.ndarray, py: np.ndarray, speed: np.ndarray):
        """-> (ball (n,2) float32, source (n,) uint8, gap_len (n,) int32, dist_to_obs (n,) float32, stats dict)."""
        cfg = self.cfg
        n = len(ball_xy)
        L, W = cfg.pitch_m
        lo = np.array([-cfg.ball_margin_m, -cfg.ball_margin_m])
        hi = np.array([L + cfg.ball_margin_m, W + cfg.ball_margin_m])
        xy = ball_xy.astype(np.float64).copy()
        stats = {"spikes_rejected": self._reject_spikes(xy) if cfg.reject_observed_spikes else 0,
                 "gaps": 0, "spline": 0, "linear": 0, "long_gaps": 0}
        obs = np.isfinite(xy).all(axis=1)
        src = np.where(obs, self.SRC_OBSERVED, self.SRC_NONE).astype(np.uint8)
        gap_len = np.zeros(n, np.int32)
        obs_idx = np.flatnonzero(obs)
        long_gaps: List[Tuple[int, int]] = []
        C = cfg.spline_context_points

        for a, b in self._gaps(~obs):
            stats["gaps"] += 1
            gap_len[a:b] = b - a
            if a == 0 or b == n or (b - a) > cfg.max_spline_gap or len(obs_idx) < 2:
                long_gaps.append((a, b))
                continue
            k = int(np.searchsorted(obs_idx, a))
            knots = np.concatenate([obs_idx[max(0, k - C):k], obs_idx[k:k + C]])
            ts = np.arange(a, b)
            fill = CubicSpline(knots, xy[knots], bc_type="natural")(ts)
            path = np.vstack([xy[a - 1], fill, xy[b]])
            steps = np.hypot(np.diff(path[:, 0]), np.diff(path[:, 1]))
            if steps.max() * cfg.fps > cfg.max_ball_speed_mps:        # spline overshoot -> straight line
                fill = np.column_stack([np.interp(ts, [a - 1, b], [xy[a - 1, j], xy[b, j]]) for j in (0, 1)])
                src[a:b] = self.SRC_LINEAR
                stats["linear"] += 1
            else:
                src[a:b] = self.SRC_SPLINE
                stats["spline"] += 1
            xy[a:b] = np.clip(fill, lo, hi)

        if long_gaps:
            stats["long_gaps"] = len(long_gaps)
            idx = np.concatenate([np.arange(a, b) for a, b in long_gaps])
            cent = self._cluster_centroids(px, py, speed, idx)
            pos = 0
            for a, b in long_gaps:
                seg = cent[pos:pos + (b - a)]
                pos += b - a
                left = xy[a - 1] if a > 0 and obs[a - 1] else None
                right = xy[b] if b < n and obs[b] else None
                seg = self._finish_cluster_segment(seg.copy(), left, right, lo, hi)
                ok = np.isfinite(seg).all(axis=1)
                xy[a:b][ok] = seg[ok]
                src[a:b][ok] = self.SRC_CLUSTER

        if obs.any():
            dist_obs = ndimage.distance_transform_edt(~obs).astype(np.float32)
        else:
            dist_obs = np.full(n, 1e4, np.float32)
        return xy.astype(np.float32), src, gap_len, dist_obs, stats


class PlayerSmoother:
    """NaN-aware Gaussian smoothing of player tracks + velocity estimation (central differences)."""

    def __init__(self, sigma: float, fps: int, max_speed: float):
        self.sigma, self.fps, self.max_speed = sigma, fps, max_speed

    def smooth(self, arr: np.ndarray) -> np.ndarray:
        return nan_gaussian(arr, self.sigma, axis=0)

    def velocity(self, sx: np.ndarray, sy: np.ndarray):
        with np.errstate(invalid="ignore"):
            vx = np.gradient(sx, axis=0) * self.fps
            vy = np.gradient(sy, axis=0) * self.fps
            spd = np.hypot(vx, vy)
            bad = spd > self.max_speed
        vx[bad] = np.nan
        vy[bad] = np.nan
        spd[bad] = np.nan
        return vx.astype(np.float32), vy.astype(np.float32), spd.astype(np.float32)

In [ ]:
@dataclass
class CleanTracking:
    """Output of preprocessing. All arrays are indexed by video frame; rows outside the graded halves are NaN / 0."""
    px: np.ndarray                 # (T, P) smoothed metres
    py: np.ndarray
    vx: np.ndarray                 # (T, P) m/s
    vy: np.ndarray
    team: np.ndarray               # (P,) 0 home / 1 away
    ball_xy: np.ndarray            # (T, 2) repaired ball track (metres)
    ball_src: np.ndarray           # (T,) uint8: 0 observed, 1 spline, 2 linear, 3 cluster, 4 none
    ball_gap_len: np.ndarray       # (T,) int32: length of the gap this frame sits in (0 = observed)
    ball_dist_obs: np.ndarray      # (T,) float32: frames to the nearest observed ball sample
    half_blocks: List[Tuple[int, int, int]]
    fps: int = 25
    pitch_m: Tuple[float, float] = FALLBACK_PITCH_M
    known_dead_ranges: List[Tuple[int, int]] = field(default_factory=list)
    config_hash: str = ""
    raw_signature: str = ""

    @property
    def n_frames(self) -> int:
        return int(self.px.shape[0])

    def half_id(self) -> np.ndarray:
        h = np.zeros(self.n_frames, np.int8)
        for half, a, b in self.half_blocks:
            h[a:b + 1] = half
        return h

    def save(self, path: Path) -> None:
        np.savez_compressed(
            path, px=self.px, py=self.py, vx=self.vx, vy=self.vy, team=self.team, ball_xy=self.ball_xy,
            ball_src=self.ball_src, ball_gap_len=self.ball_gap_len, ball_dist_obs=self.ball_dist_obs,
            half_blocks=np.asarray(self.half_blocks, np.int64), known_dead=np.asarray(self.known_dead_ranges, np.int64).reshape(-1, 2),
            fps=np.asarray(self.fps), pitch_m=np.asarray(self.pitch_m, np.float64),
            config_hash=np.asarray(self.config_hash), raw_signature=np.asarray(self.raw_signature))

    @classmethod
    def load(cls, path: Path) -> "CleanTracking":
        z = np.load(path, allow_pickle=False)
        return cls(px=z["px"], py=z["py"], vx=z["vx"], vy=z["vy"], team=z["team"], ball_xy=z["ball_xy"],
                   ball_src=z["ball_src"], ball_gap_len=z["ball_gap_len"], ball_dist_obs=z["ball_dist_obs"],
                   half_blocks=[tuple(int(v) for v in r) for r in z["half_blocks"]], fps=int(z["fps"]),
                   pitch_m=tuple(float(v) for v in z["pitch_m"]),
                   known_dead_ranges=[tuple(int(v) for v in r) for r in z["known_dead"]],
                   config_hash=str(z["config_hash"]), raw_signature=str(z["raw_signature"]))


class TrackingPreprocessor:
    """Smooth players, compute velocities, repair the ball - half by half."""

    def __init__(self, cfg: Optional[PreprocessConfig] = None):
        self.cfg = cfg or PreprocessConfig()
        self.infiller = BallInfiller(self.cfg)
        self.smoother = PlayerSmoother(self.cfg.player_sigma_frames, self.cfg.fps, self.cfg.max_player_speed_mps)
        self.diagnostics: Dict[str, dict] = {}

    def run(self, raw: TrackingData, verbose: bool = True) -> CleanTracking:
        T, P = raw.px.shape
        px = np.full((T, P), np.nan, np.float32)
        py = np.full((T, P), np.nan, np.float32)
        vx = np.full((T, P), np.nan, np.float32)
        vy = np.full((T, P), np.nan, np.float32)
        ball = np.full((T, 2), np.nan, np.float32)
        src = np.full(T, self.infiller.SRC_NONE, np.uint8)
        gap_len = np.zeros(T, np.int32)
        dist_obs = np.full(T, 1e4, np.float32)
        for half, a, b in raw.half_blocks:
            t0 = time.time()
            sl = slice(a, b + 1)
            sx, sy = self.smoother.smooth(raw.px[sl]), self.smoother.smooth(raw.py[sl])
            ux, uy, spd = self.smoother.velocity(sx, sy)
            ball_b, src_b, gap_b, dist_b, stats = self.infiller.infill_block(raw.ball_xy[sl], sx, sy, spd)
            px[sl], py[sl], vx[sl], vy[sl] = sx, sy, ux, uy
            ball[sl], src[sl], gap_len[sl], dist_obs[sl] = ball_b, src_b, gap_b, dist_b
            n = b - a + 1
            counts = {name: float((src_b == code).mean()) for name, code in
                      [("observed", 0), ("spline", 1), ("linear", 2), ("cluster", 3), ("none", 4)]}
            self.diagnostics[f"half{half}"] = {**stats, **{f"frac_{k}": v for k, v in counts.items()}}
            if verbose:
                print(f"[preprocess] half {half}: {n} frames in {time.time() - t0:.1f}s | ball observed {counts['observed']:.1%} "
                      f"-> spline {counts['spline']:.1%}, linear {counts['linear']:.1%}, cluster {counts['cluster']:.1%}, "
                      f"still missing {counts['none']:.1%} | gaps {stats['gaps']} (long {stats['long_gaps']}), "
                      f"spikes rejected {stats['spikes_rejected']}")
        return CleanTracking(px=px, py=py, vx=vx, vy=vy, team=raw.team, ball_xy=ball, ball_src=src, ball_gap_len=gap_len,
                             ball_dist_obs=dist_obs, half_blocks=list(raw.half_blocks), fps=raw.fps, pitch_m=raw.pitch_m,
                             known_dead_ranges=list(raw.known_dead_ranges),
                             config_hash=config_hash(self.cfg), raw_signature=raw.signature)


def config_hash(cfg) -> str:
    return hashlib.md5(repr(asdict(cfg)).encode()).hexdigest()[:12]


def load_or_build_clean(raw: TrackingData, cfg: Optional[PreprocessConfig] = None, force: bool = False,
                        verbose: bool = True) -> CleanTracking:
    """Cached preprocessing: artifacts/tracking_clean.npz (rebuilt if the config or the raw data changed)."""
    cfg = cfg or PreprocessConfig()
    ARTIFACT_DIR.mkdir(exist_ok=True)
    path = ARTIFACT_DIR / "tracking_clean.npz"
    if path.is_file() and not force:
        c = CleanTracking.load(path)
        if c.config_hash == config_hash(cfg) and c.raw_signature == raw.signature:
            if verbose:
                print(f"[preprocess] loaded cached {path.name}")
            return c
    c = TrackingPreprocessor(cfg).run(raw, verbose=verbose)
    c.save(path)
    return c